# Fine-Tuning CodeLlama-7B for SQL & PySpark Code Generation
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**Hardware:** Kaggle T4 GPU (15GB VRAM)  
**Base Model:** `codellama/CodeLlama-7b-Instruct-hf`

## Steps
1. Install dependencies
2. Load and prepare the dataset
3. Load CodeLlama-7B in 4-bit (QLoRA)
4. Configure LoRA adapters
5. Train with SFTTrainer
6. Evaluate and save the model
7. Push adapter weights to Hugging Face Hub

In [1]:
import os

# Check what's inside the input folder
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/rakesh17737/llm-sql-codegen-dataset/val.jsonl
/kaggle/input/datasets/rakesh17737/llm-sql-codegen-dataset/train.jsonl


## Step 1 — Install Dependencies

In [2]:
# Fix for Kaggle's CUDA 12.8 environment
!pip install -q -U bitsandbytes
!pip install -q triton==2.1.0
!pip install -q \
    transformers==4.40.0 \
    peft==0.10.0 \
    trl==0.8.6 \
    accelerate==0.29.3 \
    datasets==2.19.0 \
    huggingface_hub

print('All dependencies installed.')

^C
ERROR: Operation cancelled by user
^C
ERROR: Operation cancelled by user
^C
ERROR: Operation cancelled by user
All dependencies installed.


## Step 2 — Authenticate with Hugging Face
> Add your HF token as a Kaggle Secret named `HF_TOKEN`

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('Logged in to Hugging Face.')

Logged in to Hugging Face.


## Step 3 — Load Dataset

In [4]:
from datasets import load_dataset

# Update this path to your Kaggle dataset location
DATASET_PATH = '/kaggle/input/datasets/rakesh17737/llm-sql-codegen-dataset'

dataset = load_dataset(
    'json',
    data_files={
        'train': f'{DATASET_PATH}/train.jsonl',
        'validation': f'{DATASET_PATH}/val.jsonl'
    }
)

print(f'Train samples   : {len(dataset["train"])}')
print(f'Val samples     : {len(dataset["validation"])}')
print('\nSample prompt preview:')
print(dataset['train'][0]['text'][:300], '...')

Train samples   : 26
Val samples     : 3

Sample prompt preview:
Below is an instruction that describes a data engineering task.
Write a response that appropriately completes the request.

### Instruction:
Write PySpark code to join two DataFrames and handle null values.

### Input:
DataFrames: orders_df (order_id, customer_id, amount), customers_df (customer_id, ...


## Step 4 — Load CodeLlama-7B in 4-bit (QLoRA)

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'codellama/CodeLlama-7b-Instruct-hf'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float32,   # ← float32 not float16
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f'Model loaded on: {next(model.parameters()).device}')

Loading tokenizer...
Loading model in 4-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded on: cuda:0


## Step 5 — Configure LoRA Adapters

In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training (freezes base weights)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                        # LoRA rank — higher = more capacity, more memory
    lora_alpha=32,               # scaling factor (alpha / r = 2 is standard)
    target_modules=[             # which layers to apply LoRA to
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads() 
model.print_trainable_parameters()
# Expected: ~0.8% trainable params (only LoRA adapters train, not the full 7B)

trainable params: 39,976,960 || all params: 6,778,523,648 || trainable%: 0.5897590991188056


## Step 6 — Train with SFTTrainer

In [8]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = '/kaggle/working/codellama-sql-pyspark'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,        # ← reduced from 4
    per_device_eval_batch_size=2,         # ← reduced from 4
    gradient_accumulation_steps=4,        # ← increased from 2 (keeps effective batch=8)
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',            # ← changed from paged_adamw_8bit
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=False,                           # ← changed from True
    bf16=False,                           # ← added
    logging_steps=10,
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=lora_config,
    dataset_text_field='text',
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print('Starting training...')
trainer.train()
print('Training complete!')

2026-05-23 17:58:55.084839: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779559135.104678     824 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779559135.111014     824 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779559135.128015     824 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779559135.128032     824 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779559135.128035     824 computation_placer.cc:177] computation placer alr

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Epoch,Training Loss,Validation Loss
0,No log,1.134002
1,No log,0.752478
2,No log,0.647090


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a n

Training complete!


## Step 7 — Evaluate: Test Code Generation

In [10]:
def generate_code(instruction: str, input_context: str = '', max_new_tokens: int = 256) -> str:
    """Generate SQL or PySpark code from a natural language instruction."""
    prompt = f"""Below is an instruction that describes a data engineering task.
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input_context}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split('### Response:')[-1].strip()


# ── Test SQL generation ───────────────────────────────────────────────────────
print('TEST 1: SQL generation')
print('-' * 60)
sql_output = generate_code(
    instruction='Write a SQL query to find the top 5 products by total revenue this quarter.',
    input_context='Table: sales (product_id, product_name, sale_date, revenue)'
)
print(sql_output)

print('\nTEST 2: PySpark generation')
print('-' * 60)
pyspark_output = generate_code(
    instruction='Write PySpark code to calculate the 30-day rolling average of daily active users.',
    input_context='DataFrame: df with columns (date, daily_active_users)'
)
print(pyspark_output)

TEST 1: SQL generation
------------------------------------------------------------
SELECT product_name, SUM(revenue) AS total_revenue
FROM sales
WHERE sale_date >= '2020-01-01' AND sale_date < '2020-04-01'
GROUP BY product_name
ORDER BY total_revenue DESC
LIMIT 5;

TEST 2: PySpark generation
------------------------------------------------------------
from pyspark.sql import functions as F

df.withColumn("rolling_30_day_avg", F.avg("daily_active_users").over(Window.partitionBy("date").orderBy("date").rowsBetween(-30, 0)))


## Step 8 — Save Model & Push to Hugging Face Hub

In [9]:
# Save adapter locally
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model adapter saved to {OUTPUT_DIR}')

# Push adapter weights to HF Hub (optional but recommended for portfolio)
HF_REPO = 'Rakesh17737/codellama-sql-pyspark'   # ← update this

trainer.model.push_to_hub(HF_REPO, token=hf_token)
tokenizer.push_to_hub(HF_REPO, token=hf_token)
print(f'Model pushed to https://huggingface.co/{HF_REPO}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model adapter saved to /kaggle/working/codellama-sql-pyspark


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model pushed to https://huggingface.co/Rakesh17737/codellama-sql-pyspark
